<a href="https://colab.research.google.com/github/ysasson-portfolio/text-analytics-spring-2026/blob/main/assignment_5/notebooks/Yarden_Sasson_A5_OptionB_Job_Fit_Starter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 5 — Option B: Job Fit Analyzer
## BSAN 6200: Text Mining & Social Media Analytics — Spring 2026

**Student Name:** Yarden Sasson
**Date:** May 13, 2026  
**Option:** B — Job Fit Analyzer  
**API Path:** Paid

---

### Table of Contents
1. [Setup and Imports](#1-setup)
2. [Load Job Descriptions and Resume](#2-loading)
3. [Text Chunking](#3-chunking)
4. [Embedding and Vector Store](#4-embedding)
5. [Analysis Prompts and Chain](#5-analysis)
6. [Zero-shot vs. Few-shot Comparison](#6-comparison)
7. [Evaluation](#7-evaluation)

> **Reminder:** The Streamlit app is a separate file (`streamlit_app.py`). This notebook builds and tests the analysis pipeline.  
> See the Option B Implementation Guide for detailed step requirements.

---
<a id="1-setup"></a>
## 1. Setup and Imports

Install required packages and load your API key from a `.env` file.  
**Do NOT hardcode API keys in this notebook.**

Suggested packages: `langchain`, `langchain-openai` or `langchain-community`, `chromadb` or `faiss-cpu`, `pypdf`, `python-dotenv`, `pandas`, `sentence-transformers` (free path)

In [89]:
# ── Install packages (uncomment as needed) ──
!pip install langchain langchain-openai chromadb pypdf python-dotenv sentence-transformers
!pip install -q langchain langchain-community
# ── Load API keys from .env ──
import os
import pandas as pd
from dotenv import load_dotenv
from langchain_core.documents import Document
import requests

# ── Your imports below ──

In [90]:
#Clone the Repository (only needs to happen once)
!git clone https://github.com/ysasson-portfolio/text-analytics-spring-2026.git

fatal: destination path 'text-analytics-spring-2026' already exists and is not an empty directory.


In [91]:
#If the github gets updated and I want the latest changes without re-cloning
%cd /content/text-analytics-spring-2026
!git pull

/content/text-analytics-spring-2026
Already up to date.


---
<a id="2-loading"></a>
## 2. Load Job Descriptions and Resume

**Required:**
- 10+ JD files in `data/job_descriptions/` (each as a separate .txt or .pdf)
- Your resume in `data/resume/`
- A metadata file `data/jd_metadata.csv` with columns: filename, company, title, source_url, date_collected

Print: number of JDs loaded, number of resume docs, and preview content from each.

In [92]:
# ── Load JD metadata ──
metadata_df = pd.read_csv("https://raw.githubusercontent.com/ysasson-portfolio/text-analytics-spring-2026/refs/heads/main/assignment_5/data/jd_metadata.csv")

print(metadata_df)

                                           Job Title  \
0                      Business Intelligence Analyst   
1  Business Intelligence Analyst, Sports - Brand ...   
2                      Business Intelligence Analyst   
3                                   Business Analyst   
4                                   Business Analyst   
5                                   Business Analyst   
6                                 Business Analyst I   
7                Sr. Analyst, Strategy and Analytics   
8                                 Strategy Associate   
9                                  Manager, Strategy   

                                    Company  \
0                             Guitar Center   
1                   Creative Artists Agency   
2  Los Angeles Tourism and Convention Board   
3             Red Bull Distribution Company   
4                                   Hadrian   
5                        Polestar Analytics   
6                  Skyworks Solutions, Inc.   
7      

In [93]:
#Use the folder path that is directly connected to the job description folder
folder_path = "/content/text-analytics-spring-2026/assignment_5/data/job_descriptions/"

#Empty list to store the descriptions and the metadata
job_descriptions=[]

for index, row in metadata_df.iterrows():
    #Pull the file name from the metadata dataframe
    filename = row["File Name"] + ".txt"
    #Create the final path using the folder path and file name
    file_path = os.path.join(folder_path, filename)

    #Create the document with the following metadata from the dataframe
    document = Document(
        page_content=open(file_path, "r", encoding="utf-8").read(),
        metadata={
            "File Name": row["File Name"],
            "Company": row["Company"],
            "Job Title": row["Job Title"],
            "Source URL": row["Source URL"],
            "Date Collected": row["Date Collected"],
            "Document Type": "Job Description"
        }
    )
    #Add the job description to the empty list
    job_descriptions.append(document)


print(f"Number of JDs loaded: {len(job_descriptions)}")

Number of JDs loaded: 10


In [94]:
#Show the sample of the metadata and the job description generated
print(job_descriptions[0].metadata)
print(job_descriptions[0].page_content)

{'File Name': 'Business Inteligence Analyst-Guitar Center', 'Company': 'Guitar Center', 'Job Title': 'Business Intelligence Analyst', 'Source URL': 'https://www.linkedin.com/jobs/collections/recommended/?currentJobId=4384414218&origin=JYMBII_IN_APP_NOTIFICATION&originToLandingJobPostings=4392357648%2C4407124320', 'Date Collected': '4/28/2026', 'Document Type': 'Job Description'}
About the Role:  
At Guitar Center, the Data Team is deploying data to inform and empower the company with insight, to drive customer success and business value. We are looking for a Business Intelligence Analyst to help advance that vision as part of the Business Intelligence team, which is part of the larger central Data Organization. You will work with stakeholders and data peers to create a vibrant ecosystem that enables data self-service across the company. This is a chance to affect millions of musicians at the United States' largest musical retailer, and your success as part of a world-class analytics or

In [95]:
#Use the folder path that is directly connected to the resume
resume_path = "https://raw.githubusercontent.com/ysasson-portfolio/text-analytics-spring-2026/refs/heads/main/assignment_5/data/resume/resume.txt"

resume_text= requests.get(resume_path).text

#Create the document with the following metadata that we supplied
resume = Document(
    page_content=resume_text,
    metadata_df={
        "File Name": "resume.txt",
        "Document Type": "Resume"}
)

print("Resume has been loaded")
print(resume.page_content)

Resume has been loaded
Yarden Sasson


EDUCATION

Loyola Marymount University                                                        
Masters of Science in Business Analytics                                                                                         Class of 2026
•	Relevant Coursework: Data Management for Business Intelligence, Introduction to Machine Learning, Strategic Integration Analytics
•	Honors Society: Beta Gamma Sigma

University of California, Los Angeles (UCLA)                                                        
Bachelors of Science in Cognitive Science with a Specialization in Computing                               Class of 2020
•	Relevant Coursework: Advanced Topics in MATLAB Programming for Behavioral Sciences, Science of Language, and Neural Networks
•	Activities: Den Operations Club, UCLA’s The Den, UCLA Hillel

WORK EXPERIENCE

Israel Economic and Trade Mission to the West Coast                                                              
Head of Inn

In [96]:
# ── Preview sample content ──


---
<a id="3-chunking"></a>
## 3. Text Chunking

Split your documents into chunks.  
**Required:** Try at least 2 chunking strategies, compare them quantitatively, and justify your final choice.

**Hint:** JDs often have natural sections (Requirements, Responsibilities, Qualifications). Consider whether your splitter respects these boundaries.

In [97]:
#Combine all documents so we can chunk them effectively
every_document= job_descriptions + [resume]

In [98]:
# ── Strategy 1 ──
def chunk_text(text, chunk_size=300, overlap=50):
    """Split text into overlapping chunks."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk.strip())
        start += chunk_size - overlap
    return [c for c in chunks if len(c) > 20]  # skip tiny fragments

fixed_chunks = []
#Apply the strategy to each document and cumulatively counting the number of chunks while also
for doc in every_document:
    chunks = chunk_text(doc.page_content, chunk_size=400, overlap=100)

    for i, chunk in enumerate(chunks):
        fixed_chunks.append({
            "Text": chunk,
            "File Name": doc.metadata.get("File Name", ""),
            "Company": doc.metadata.get("Company", ""),
            "Job Title": doc.metadata.get("Job Title", ""),
            "Document Type": doc.metadata.get("Document Type", ""),
            "Chunk_id": i,
            "Strategy": "Fixed Size"
        })


print("Strategy 1 chunks:", len(fixed_chunks))
print(f"\nChunks per source:")
from collections import Counter
for job_title, count in Counter(c.get("Company", "Resume") or "Resume"
        for c in fixed_chunks).items():
    print(f"  {job_title}: {count}")


Strategy 1 chunks: 155

Chunks per source:
  Guitar Center: 10
  Creative Artists Agency: 10
  Los Angeles Tourism and Convention Board: 27
  Red Bull Distribution Company: 13
  Hadrian: 18
  Polestar Analytics: 13
  Skyworks Solutions, Inc.: 8
  SoFi Stadium and Hollywood Park: 24
  Cedars Sinai: 12
  Paramount: 8
  Resume: 12


In [99]:
# ── Strategy 2 ──
def chunk_text_by_sentences(text, chunk_size=400, overlap_words=20):
    """Split text into chunks that respect sentence boundaries."""
    import re
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())

    chunks = []
    current_chunk = ""

    for sentence in sentences:
        if len(current_chunk) + len(sentence) > chunk_size and current_chunk:
            chunks.append(current_chunk.strip())
            # Keep overlap by taking the end of the current chunk
            words = current_chunk.split()
            overlap_text = " ".join(words[-overlap_words:]) if len(words) > overlap_words else current_chunk
            current_chunk = overlap_text + " " + sentence
        else:
            current_chunk += (" " if current_chunk else "") + sentence

    if current_chunk.strip():
        chunks.append(current_chunk.strip())

    return chunks

sentence_chunks = []

#Apply the strategy to each document while cumulatively counting the chunks and storing the relavent metadata within each chunk
for doc in every_document:
    chunks = chunk_text_by_sentences(doc.page_content, chunk_size=400, overlap_words=10)

    for i, chunk in enumerate(chunks):
        sentence_chunks.append({
            "Text": chunk,
            "File Name": doc.metadata.get("File Name", ""),
            "Company": doc.metadata.get("Company", ""),
            "Job Title": doc.metadata.get("Job Title", ""),
            "Document Type": doc.metadata.get("Document Type", ""),
            "Chunk_id": i,
            "Strategy": "Sentence Aware"
        })

print("Strategy 2 chunks:", len(sentence_chunks))
print(f"\nChunks per source:")
from collections import Counter
for job_title, count in Counter(c.get("Company", "Resume") or "Resume"
        for c in sentence_chunks).items():
    print(f"  {job_title}: {count}")

Strategy 2 chunks: 143

Chunks per source:
  Guitar Center: 11
  Creative Artists Agency: 9
  Los Angeles Tourism and Convention Board: 29
  Red Bull Distribution Company: 14
  Hadrian: 9
  Polestar Analytics: 15
  Skyworks Solutions, Inc.: 7
  SoFi Stadium and Hollywood Park: 26
  Cedars Sinai: 14
  Paramount: 4
  Resume: 5


### Chunking Decision

Looking at the overall size of the job descriptions and the resume, we can see that the size in terms of length varies. There are some such as the Paramount and the Skyworks Solutions job descriptions are shorter while the LA Tourism and Convention Board and SoFi Stadium and Hollywood Park have longer descriptions. This is why we need to have a cluster size that is small enough to maintain the meaning of the text along with the context, while having a large enough cluster size that produces multiple clusters.

After messing around with the cluster sizes to the number of clusters that were produced, I believed that the best results were produced at a cluster size of 400. Even though some of the job descriptions ended up with a higher number of clusters, there are enough clusters for it to perform well enough with the LLM model that will eventually be chosen.

**Which strategy did you choose? Why?**

In the end, I chose a sentence based chunking strategy (Strategy 2). The sentence based method produced a smaller amount of clusters. The fixed-chunk method produced a total of 155 based on the all job descriptions and the resume, while the sentence based method produced 143 clusters. That is a difference of 9.09% between the two. I also decided to use this method because it will most likely acheive better context within the chunks. Since this strategy considers the whole word in the overlap instead of the character count, it will not end the cluster in the middle of word giving us words that do not exist. When looking at the amount of overlap, I wanted to make sure that there would be enough words to make sure that context and semantic meaning would be understood by the model. With shorter job descriptions I wanted to make sure that the overlap would not be as drastic to lower the number of clusters or give it too much information per cluster that would cause the model to overfit.

**Final settings (chunk_size, overlap):**
Chunk Size: 400 characters
Overlap Words: 10 words

---
<a id="4-embedding"></a>
## 4. Embedding and Vector Store

Embed your chunks and store them in a vector database (ChromaDB or FAISS).

**Paid path:** OpenAI `text-embedding-3-small`  
**Free path:** `sentence-transformers/all-MiniLM-L6-v2`

After creating the store, run a test similarity search to verify it works.

In [101]:
from sentence_transformers import SentenceTransformer
import chromadb

# Load embedding model
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

# Check embedding size
sample = embed_model.encode("What job is this resume a good fit for?")
print(f"Embedding dimensions: {len(sample)}")
print(f"First 10 values: {sample[:10].tolist()}")

# Create ChromaDB client
chroma_client = chromadb.Client()

# Create or reset collection
collection = chroma_client.get_or_create_collection(
    name="job_fit_sentence_chunks",
    metadata={"hnsw:space": "cosine"}
)

# Pull text and metadata from your sentence-based chunks
documents = [c["Text"] for c in sentence_chunks]

metadatas = [
    {
        "job_title": c.get("Job Title", "Resume") or "Resume",
        "company": c.get("Company", "Resume") or "Resume",
        "source": c.get("Job Title", "Resume") or "Resume"
    }
    for c in sentence_chunks
]

ids = [f"sentence_chunk_{i}" for i in range(len(sentence_chunks))]

# Create embeddings yourself using SentenceTransformer
embeddings = embed_model.encode(documents).tolist()

# Add chunks to ChromaDB
collection.add(
    documents=documents,
    embeddings=embeddings,
    metadatas=metadatas,
    ids=ids
)

print(f"Vector store created with {collection.count()} sentence-based chunks")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding dimensions: 384
First 10 values: [-0.11932414770126343, 0.05582309886813164, -0.007771473843604326, 0.03355583921074867, -0.012139877304434776, 0.05592120438814163, -0.002076872391626239, -0.004593976773321629, -0.1080927923321724, -0.06075459346175194]
Vector store created with 143 sentence-based chunks


In [104]:
# ── Verify: run a test similarity search ──

def search(query, k=3):
    """Search the vector store and return top-k results."""
    results = collection.query(
        query_texts=[query],
        n_results=k
    )
    # ChromaDB returns nested lists, so we unpack
    docs = results["documents"][0]
    sources = [m["source"] for m in results["metadatas"][0]]
    distances = results["distances"][0]
    return list(zip(docs, sources, distances))


# ── Test: technology question ──
query = "Which jobs require sql?"
print(f'Query: "{query}"\n')

for i, (text, source, dist) in enumerate(search(query, k=3)):
    print(f"Result {i+1} [Source: {source}] (distance: {dist:.3f}):")
    print(f"  {text[:400]}")
    print()

Query: "Which jobs require sql?"

Result 1 [Source: Sr. Analyst, Strategy and Analytics] (distance: 0.472):
  visualization tools (Tableau and Google Looker expertise is a plus). Advanced proficiency with ETL and other data workflow tools (Alteryx, Matillion, etc). Proficient with common data languages (SQL, Python, R, etc). Experience using workflows for automation purposes is a plus. Proficiency with Ticketmaster / Archtics Data Platform preferred.

Result 2 [Source: Business Intelligence Analyst] (distance: 0.504):
  of relevant experience is required. High school diploma or equivalent. Must be willing to partake in a comprehensive background check including a drug test in accordance with applicable laws. Proficiency with Domo or other Business Intelligence tools (i.e., Tableau, Power BI). Proficiency in SQL and Python programming. Data architecture design and data preparation experience.

Result 3 [Source: Business Analyst] (distance: 0.512):
  Business, Economics, Information Syst

After embedding the infomration, I ran a similarity test by asking "What jobs require SQL?". After looking at the responses I see that responses have a moderate score after using a cosine similarity score. The responses all showed that a SQL proficiency was necessary for the job.

The first results produced a stronger than moderate result with a score 0.472. It was the one where SQL appears earlier and requires a strong proficiency. The second result showed a strong profieciency as well later on in the description. This had a slightly worse score than the first on at 0.504. The third one did not identify SQL as a proficiency; however, it asked for people that had experience in relational databases such as SQL. This is performing with a similarity score of 0.512.

These three scores being sclose together showed me that the chunks found were relevant to the query I input into the function. It also showed that it captured the meaning behind what I asked for.

---
<a id="5-analysis"></a>
## 5. Analysis Prompts and Chain

Build 3 analysis types, each with its own prompt (one iteration for 3 types) or 3 iterations for 1 type with its own prompt:

1. **Skill Gap Report:** Compare resume skills vs. JD requirements. Output matching skills, missing skills, and recommended actions.
2. **Keyword Alignment:** Extract key terms from a JD, check which appear in the resume, report a match rate.
3. **Fit Summary:** 3-4 sentence narrative assessment citing evidence from both documents.

You also need to wire up the LLM and a way to pass a specific JD + resume into each prompt.

**Required:** Document at least 3 prompt iterations total (across any analysis type) with rationale.

**Reminder:** Prompt design must be your own work (Tier 2 — AI prohibited for this step).

In [73]:
# ── Initialize LLM ──
from openai import OpenAI

load_dotenv(".env)")

# ── Create the client ──

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

print("LLM initialized")


In [ ]:
def skill_gap_analysis(question, k=5):

    # Retrieve
    query_embedding = embed_model.encode(question).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    # Build context
    context = "\n\n".join(results["documents"][0])

    # Build prompt
    prompt = f"""
Use the context to answer the question.

Context:
{context}

Question:
{question}

Answer:
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt},
                  {"role":"system" , "content": "You find the skill gaps of a person by comparing the job descriptions to resumes using only the documents that are provided"}],
        temperature=0.1
    )
    answer = response.choices[0].message.content
    return {
    "answer": answer,
    "sources": sources,
    "context": context,
    "prompt": full_prompt
}

In [74]:
# ── Analysis 1: Skill Gap Report ──
#Give me a analysis of the skill gap between the resume inputted and the job description (Baseline)
#Compare and Contrast the resume to the job posting and let me know where the skill gap needs to improve (Iteration 1)
#Explain what is missing from my resume and why it is important for this position (Iteration 2)
#If I have the skills for this position, do I have enough experience for the position as well? (Iteration 3) (Potentially add more and be aware of the changes)

In [75]:
# ── Analysis 2: Keyword Alignment ──
def keyword_alignment_analysis(question, k=5):

    # Retrieve
    query_embedding = embed_model.encode(question).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    # Build context
    context = "\n\n".join(results["documents"][0])

    # Build prompt
    prompt = f"""
Use the context to answer the question.

Context:
{context}

Question:
{question}

Answer:
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt},
                  {"role":"system" , "content": "You find the relevant positions by comparing keywords in the job description to the resume"}],
        temperature=0.1
    )
    answer = response.choices[0].message.content
    return {
    "answer": answer,
    "sources": sources,
    "context": context,
    "prompt": full_prompt
}

In [76]:
# ── Analysis 3: Fit Summary ──
def fit_summary_analysis(question, k=5):

    # Retrieve
    query_embedding = embed_model.encode(question).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    # Build context
    context = "\n\n".join(results["documents"][0])

    # Build prompt
    prompt = f"""
Use the context to answer the question.

Context:
{context}

Question:
{question}

Answer:
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt},
                  {"role":"system" , "content": "You evaluate if a candidate's fit for the role based on the job descriptions and the resume of the candidate."}],
        temperature=0.1
    )
    answer = response.choices[0].message.content
    return {
    "answer": answer,
    "sources": sources,
    "context": context,
    "prompt": full_prompt
}

### Prompt Iteration Log

Document at least 3 total iterations across any of the analysis types.

**Iteration 1:** [Which analysis? What changed? Why? What improved?]

**Iteration 2:** [Which analysis? What changed? Why? What improved?]

**Iteration 3:** [Which analysis? What changed? Why? What improved?]

---
<a id="6-comparison"></a>
## 6. Zero-shot vs. Few-shot Comparison

Pick one of your 3 analysis types. Create a few-shot version by adding 1-2 example input/output pairs to the prompt. Run both versions on the same JD and compare outputs.

**Reminder:** You must write the few-shot examples yourself (Tier 2).

In [77]:
# ── Few-shot version of your chosen analysis ──
# Based on the following job description, identify whether the job is match from the resume.

#Job Description: (insert job description here)
#Match: Yes because ....

#Job Description: (insert job description here)
#Match: No because ....

In [78]:
# ── Run both on the same JD, display side by side ──


### Zero-shot vs. Few-shot Analysis

**Which analysis type did you compare?**

**Which performed better?**

**Why? (use specific examples from the outputs above)**

---
<a id="7-evaluation"></a>
## 7. Evaluation

Run all 3 analysis types on your **top 3 target JDs** (9 total analyses).

For each, score:
- **Retrieval relevance:** Did it pull the right JD sections? (Yes/Partial/No)
- **Skill identification accuracy:** Are identified skills/gaps correct? (count correct vs. incorrect)
- **Actionability:** Are recommendations specific and useful? (1-5)
- **Faithfulness:** Does output stick to document content? (Faithful/Partial/Hallucinated)

**Reminder:** Evaluation must be your own work (Tier 2 — AI prohibited).

In [79]:
# ── Run 9 analyses (3 JDs x 3 analysis types) ──


In [80]:
# ── Summarize evaluation results ──


### Evaluation Analysis

**Which analysis type worked best?**

**Which JDs produced the best/worst results? Why?**

**Where did the system hallucinate or produce inaccurate results?**

**What would you improve?**

---

## Next Steps

1. Build your Streamlit app (`streamlit_app.py`) using the pipeline from this notebook
2. Write your Technical Manager Memo (`memo.md`)
3. Complete your AI Usage Log (`ai_log.md`)
4. Verify GitHub repository structure and commit count

---
*BSAN 6200 | Spring 2026 | Assignment 5 — Option B*